In [1]:
#Practical 10: Mini Project - End-to-End Data Engineering Solution
#Design and implement an end-to-end data engineering solution including:
#1. Data ingestion
#2. Data cleaning
#3. Data transformation
#4. Data storage
#5. Data warehouse design
#6. Reporting

In [3]:
import os
import sqlite3
import pandas as pd


In [4]:
# ============================================================
# 1. DATA INGESTION
# ============================================================

print("=" * 60)
print("PRACTICAL 10 - END-TO-END DATA ENGINEERING MINI PROJECT")
print("=" * 60)

os.makedirs("data", exist_ok=True)
os.makedirs("output", exist_ok=True)

# Sample source data
customers_data = {
    "Customer_ID": [101, 102, 103, 104, 105],
    "Customer_Name": ["Amit", "Riya", "Neha", "Rahul", "Sneha"],
    "City": ["Mumbai", "Pune", "Delhi", "Mumbai", None]
}

products_data = {
    "Product_ID": [1, 2, 3, 4, 5],
    "Product_Name": ["Laptop", "Phone", "Headphones", "Keyboard", "Mouse"],
    "Category": ["Electronics", "Electronics", "Accessories", "Accessories", "Accessories"],
    "Price": [60000, 30000, 2000, 1500, 800]
}

orders_data = {
    "Order_ID": [1001, 1002, 1003, 1004, 1005, 1005],
    "Customer_ID": [101, 102, 103, 104, 105, 105],
    "Product_ID": [1, 2, 3, 4, 5, 5],
    "Quantity": [1, 2, 3, 2, 4, 4],
    "Order_Date": [
        "2026-09-01",
        "2026-09-02",
        "2026-09-03",
        "2026-09-04",
        "2026-09-05",
        "2026-09-05"
    ]
}

payments_data = {
    "Payment_ID": [501, 502, 503, 504, 505, 506],
    "Order_ID": [1001, 1002, 1003, 1004, 1005, 1005],
    "Payment_Mode": ["UPI", "Card", "Cash", "UPI", "Card", "Card"],
    "Payment_Status": ["Paid", "Paid", "Paid", "Pending", "Paid", "Paid"]
}

# Write source files
pd.DataFrame(customers_data).to_csv("data/customers.csv", index=False)
pd.DataFrame(products_data).to_csv("data/products.csv", index=False)
pd.DataFrame(orders_data).to_csv("data/orders.csv", index=False)
pd.DataFrame(payments_data).to_csv("data/payments.csv", index=False)

print("\nSource files created successfully.")


PRACTICAL 10 - END-TO-END DATA ENGINEERING MINI PROJECT

Source files created successfully.


In [5]:
# ============================================================
# 2. EXTRACT / INGEST DATA
# ============================================================

customers = pd.read_csv("data/customers.csv")
products = pd.read_csv("data/products.csv")
orders = pd.read_csv("data/orders.csv")
payments = pd.read_csv("data/payments.csv")

print("\n--- Customers ---")
print(customers)

print("\n--- Products ---")
print(products)

print("\n--- Orders ---")
print(orders)

print("\n--- Payments ---")
print(payments)


--- Customers ---
   Customer_ID Customer_Name    City
0          101          Amit  Mumbai
1          102          Riya    Pune
2          103          Neha   Delhi
3          104         Rahul  Mumbai
4          105         Sneha     NaN

--- Products ---
   Product_ID Product_Name     Category  Price
0           1       Laptop  Electronics  60000
1           2        Phone  Electronics  30000
2           3   Headphones  Accessories   2000
3           4     Keyboard  Accessories   1500
4           5        Mouse  Accessories    800

--- Orders ---
   Order_ID  Customer_ID  Product_ID  Quantity  Order_Date
0      1001          101           1         1  2026-09-01
1      1002          102           2         2  2026-09-02
2      1003          103           3         3  2026-09-03
3      1004          104           4         2  2026-09-04
4      1005          105           5         4  2026-09-05
5      1005          105           5         4  2026-09-05

--- Payments ---
   Payment_I

In [6]:
# ============================================================
# 3. DATA CLEANING
# ============================================================

print("\n" + "=" * 60)
print("DATA CLEANING")
print("=" * 60)

# Fill missing city
customers["City"] = customers["City"].fillna("Unknown")

# Remove duplicate orders and payments
orders = orders.drop_duplicates()
payments = payments.drop_duplicates()

# Remove invalid quantities
orders = orders[orders["Quantity"] > 0].copy()

# Convert order date to datetime
orders["Order_Date"] = pd.to_datetime(orders["Order_Date"])

# Standardize text fields
customers["Customer_Name"] = customers["Customer_Name"].str.title()
customers["City"] = customers["City"].str.title()
payments["Payment_Status"] = payments["Payment_Status"].str.title()

print("\nCleaned Customers:")
print(customers)

print("\nCleaned Orders:")
print(orders)

print("\nCleaned Payments:")
print(payments)



DATA CLEANING

Cleaned Customers:
   Customer_ID Customer_Name     City
0          101          Amit   Mumbai
1          102          Riya     Pune
2          103          Neha    Delhi
3          104         Rahul   Mumbai
4          105         Sneha  Unknown

Cleaned Orders:
   Order_ID  Customer_ID  Product_ID  Quantity Order_Date
0      1001          101           1         1 2026-09-01
1      1002          102           2         2 2026-09-02
2      1003          103           3         3 2026-09-03
3      1004          104           4         2 2026-09-04
4      1005          105           5         4 2026-09-05

Cleaned Payments:
   Payment_ID  Order_ID Payment_Mode Payment_Status
0         501      1001          UPI           Paid
1         502      1002         Card           Paid
2         503      1003         Cash           Paid
3         504      1004          UPI        Pending
4         505      1005         Card           Paid
5         506      1005         Card     

In [7]:
# ============================================================
# 4. DATA TRANSFORMATION
# ============================================================

print("\n" + "=" * 60)
print("DATA TRANSFORMATION")
print("=" * 60)

# Join orders with products
sales = orders.merge(
    products,
    on="Product_ID",
    how="left"
)

# Calculate total amount
sales["Total_Amount"] = sales["Quantity"] * sales["Price"]

# Add payment information
sales = sales.merge(
    payments[["Order_ID", "Payment_Mode", "Payment_Status"]],
    on="Order_ID",
    how="left"
)

# Add customer information
sales = sales.merge(
    customers[["Customer_ID", "Customer_Name", "City"]],
    on="Customer_ID",
    how="left"
)

# Select final transformed columns
sales = sales[
    [
        "Order_ID",
        "Order_Date",
        "Customer_ID",
        "Customer_Name",
        "City",
        "Product_ID",
        "Product_Name",
        "Category",
        "Quantity",
        "Price",
        "Total_Amount",
        "Payment_Mode",
        "Payment_Status"
    ]
]

print("\nTransformed Sales Data:")
print(sales)



DATA TRANSFORMATION

Transformed Sales Data:
   Order_ID Order_Date  Customer_ID Customer_Name     City  Product_ID  \
0      1001 2026-09-01          101          Amit   Mumbai           1   
1      1002 2026-09-02          102          Riya     Pune           2   
2      1003 2026-09-03          103          Neha    Delhi           3   
3      1004 2026-09-04          104         Rahul   Mumbai           4   
4      1005 2026-09-05          105         Sneha  Unknown           5   
5      1005 2026-09-05          105         Sneha  Unknown           5   

  Product_Name     Category  Quantity  Price  Total_Amount Payment_Mode  \
0       Laptop  Electronics         1  60000         60000          UPI   
1        Phone  Electronics         2  30000         60000         Card   
2   Headphones  Accessories         3   2000          6000         Cash   
3     Keyboard  Accessories         2   1500          3000          UPI   
4        Mouse  Accessories         4    800          3200  

In [8]:
# ============================================================
# 5. DATA WAREHOUSE DESIGN
# ============================================================

print("\n" + "=" * 60)
print("DATA WAREHOUSE DESIGN")
print("=" * 60)

# Star schema:
# Dimension tables:
#   dim_customer
#   dim_product
#   dim_date
# Fact table:
#   fact_sales

dim_customer = customers[
    ["Customer_ID", "Customer_Name", "City"]
].drop_duplicates()

dim_product = products[
    ["Product_ID", "Product_Name", "Category", "Price"]
].drop_duplicates()

dim_date = sales[["Order_Date"]].drop_duplicates().copy()
dim_date["Date_ID"] = dim_date["Order_Date"].dt.strftime("%Y%m%d").astype(int)
dim_date["Year"] = dim_date["Order_Date"].dt.year
dim_date["Month"] = dim_date["Order_Date"].dt.month
dim_date["Day"] = dim_date["Order_Date"].dt.day

dim_date = dim_date[
    ["Date_ID", "Order_Date", "Year", "Month", "Day"]
]

fact_sales = sales.copy()
fact_sales["Date_ID"] = fact_sales["Order_Date"].dt.strftime("%Y%m%d").astype(int)

fact_sales = fact_sales[
    [
        "Order_ID",
        "Date_ID",
        "Customer_ID",
        "Product_ID",
        "Quantity",
        "Total_Amount",
        "Payment_Status"
    ]
]

print("\nDimension - Customer:")
print(dim_customer)

print("\nDimension - Product:")
print(dim_product)

print("\nDimension - Date:")
print(dim_date)

print("\nFact - Sales:")
print(fact_sales)


DATA WAREHOUSE DESIGN

Dimension - Customer:
   Customer_ID Customer_Name     City
0          101          Amit   Mumbai
1          102          Riya     Pune
2          103          Neha    Delhi
3          104         Rahul   Mumbai
4          105         Sneha  Unknown

Dimension - Product:
   Product_ID Product_Name     Category  Price
0           1       Laptop  Electronics  60000
1           2        Phone  Electronics  30000
2           3   Headphones  Accessories   2000
3           4     Keyboard  Accessories   1500
4           5        Mouse  Accessories    800

Dimension - Date:
    Date_ID Order_Date  Year  Month  Day
0  20260901 2026-09-01  2026      9    1
1  20260902 2026-09-02  2026      9    2
2  20260903 2026-09-03  2026      9    3
3  20260904 2026-09-04  2026      9    4
4  20260905 2026-09-05  2026      9    5

Fact - Sales:
   Order_ID   Date_ID  Customer_ID  Product_ID  Quantity  Total_Amount  \
0      1001  20260901          101           1         1         600

In [9]:
# ============================================================
# 6. DATA STORAGE - SQLITE DATA WAREHOUSE
# ============================================================

print("\n" + "=" * 60)
print("LOADING DATA INTO SQLITE DATA WAREHOUSE")
print("=" * 60)

database_path = "output/data_warehouse.db"

conn = sqlite3.connect(database_path)

dim_customer.to_sql("dim_customer", conn, if_exists="replace", index=False)
dim_product.to_sql("dim_product", conn, if_exists="replace", index=False)
dim_date.to_sql("dim_date", conn, if_exists="replace", index=False)
fact_sales.to_sql("fact_sales", conn, if_exists="replace", index=False)

print(f"\nData warehouse created: {database_path}")


LOADING DATA INTO SQLITE DATA WAREHOUSE

Data warehouse created: output/data_warehouse.db


In [10]:
# ============================================================
# 7. REPORTING
# ============================================================

print("\n" + "=" * 60)
print("REPORTING")
print("=" * 60)

# Category-wise sales report
category_report = pd.read_sql_query(
    """
    SELECT
        p.Category,
        SUM(f.Quantity) AS Total_Quantity,
        ROUND(SUM(f.Total_Amount), 2) AS Total_Sales
    FROM fact_sales f
    JOIN dim_product p
        ON f.Product_ID = p.Product_ID
    GROUP BY p.Category
    ORDER BY Total_Sales DESC;
    """,
    conn
)

print("\nCategory-wise Sales Report:")
print(category_report)

# Customer-wise sales report
customer_report = pd.read_sql_query(
    """
    SELECT
        c.Customer_Name,
        c.City,
        ROUND(SUM(f.Total_Amount), 2) AS Total_Spending
    FROM fact_sales f
    JOIN dim_customer c
        ON f.Customer_ID = c.Customer_ID
    GROUP BY c.Customer_ID, c.Customer_Name, c.City
    ORDER BY Total_Spending DESC;
    """,
    conn
)

print("\nCustomer-wise Sales Report:")
print(customer_report)

# Save reports
category_report.to_csv("output/category_sales_report.csv", index=False)
customer_report.to_csv("output/customer_sales_report.csv", index=False)




REPORTING

Category-wise Sales Report:
      Category  Total_Quantity  Total_Sales
0  Electronics               3     120000.0
1  Accessories              13      15400.0

Customer-wise Sales Report:
  Customer_Name     City  Total_Spending
0          Amit   Mumbai         60000.0
1          Riya     Pune         60000.0
2         Sneha  Unknown          6400.0
3          Neha    Delhi          6000.0
4         Rahul   Mumbai          3000.0


In [11]:
# ============================================================
# 8. DATA QUALITY CHECK
# ============================================================

print("\n" + "=" * 60)
print("DATA QUALITY CHECK")
print("=" * 60)

print("Missing values in fact table:")
print(fact_sales.isnull().sum())

print("\nDuplicate fact records:")
print(fact_sales.duplicated().sum())

print("\nTotal fact records:", len(fact_sales))



DATA QUALITY CHECK
Missing values in fact table:
Order_ID          0
Date_ID           0
Customer_ID       0
Product_ID        0
Quantity          0
Total_Amount      0
Payment_Status    0
dtype: int64

Duplicate fact records:
1

Total fact records: 6


In [12]:
# ============================================================
# 9. VERIFY DATA WAREHOUSE
# ============================================================

print("\n" + "=" * 60)
print("DATA WAREHOUSE TABLES")
print("=" * 60)

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

print(tables)

conn.close()

print("\n" + "=" * 60)
print("ETL PIPELINE COMPLETED SUCCESSFULLY")
print("=" * 60)
print("\nGenerated files:")
print("1. output/data_warehouse.db")
print("2. output/category_sales_report.csv")
print("3. output/customer_sales_report.csv")



DATA WAREHOUSE TABLES
           name
0  dim_customer
1   dim_product
2      dim_date
3    fact_sales

ETL PIPELINE COMPLETED SUCCESSFULLY

Generated files:
1. output/data_warehouse.db
2. output/category_sales_report.csv
3. output/customer_sales_report.csv
